In [1]:
import sys
# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [2]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger

c:\Users\nikit\anaconda3\envs\LLM\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


#### 1. Задаём конфигурацию графа знаний

In [3]:
# CONFIG FOR REMOTE STORAGE

remote_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='neo4j', db_config=GraphDBConnectionConfig(uri="bolt://localhost:7687", params={'user': "neo4j", 'pwd': 'password', 'db_name': 'testing'}))), # TO CHANGE
    
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing', db_name='vectorized_nodes', is_exist=True, need_to_clear=True)), # TO CHANGE
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing', db_name='vectorized_triplets', is_exist=True, need_to_clear=True)), # TO CHANGE
        embedder_config=EmbedderModelConfig(model_name_or_path='intfloat/multilingual-e5-small')),
    
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='astar', # TO CHANGE
            retriever_config=AStarGraphSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)), #KeyValueDriverConfig(db_vendor='aerospike', db_config=KVDBConnectionConfig(host='aerospikelservice', port=3000))),
        answer_generator_config=QALLMGeneratorConfig()),
    
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(),
        updator_config=LLMUpdatorConfig()),
    
    log=Logger('log/main'))

In [3]:
# CONFIG FOR IN-MEMORY STORAGE

inmemory_kg_config = RemoteKnowledgeGraphConfig(
    
    graph_struct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)), # TO CHANGE
    
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing', db_name='vectorized_nodes', is_exist=True, need_to_clear=True)), # TO CHANGE
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing', db_name='vectorized_triplets', is_exist=True, need_to_clear=True)), # TO CHANGE
        embedder_config=EmbedderModelConfig(model_name_or_path='../../models/intfloat/multilingual-e5-small')),
    
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='astar', # TO CHANGE
            retriever_config=AStarGraphSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)), # TO CHANGE
        answer_generator_config=QALLMGeneratorConfig()),
    
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(),
        updator_config=LLMUpdatorConfig()),
    
    log=Logger('log/main'))

#### 2. Инициализируем граф знаний

In [4]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [5]:
# ATTENTION !!!
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

[]

#### 3. Добавляем в граф информацию

In [5]:
rkg_main.update_memory([
    "Mikhail Menshchikov is currently a second-year master's student at ITMO.",
    "Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'",
    "Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University",
    "Petrozavodsk State University is where Mikhail Menshchikov received his bachelor's degree.",
    "Mikhail Menshchikov studied at Petrozavodsk State University and received a bachelor's degree."])

100%|██████████| 8/8 [00:00<00:00, 52510.85it/s]
Insert of existing embedding ID: 0caadc9e921bc42b049ee99dce67f49a
Add of existing embedding ID: 0caadc9e921bc42b049ee99dce67f49a
100%|██████████| 10/10 [00:00<00:00, 73973.62it/s]
Insert of existing embedding ID: 0caadc9e921bc42b049ee99dce67f49a
Insert of existing embedding ID: 0721dba30ac0399878961e2afa1ba284
Add of existing embedding ID: 0caadc9e921bc42b049ee99dce67f49a
Add of existing embedding ID: 0721dba30ac0399878961e2afa1ba284
100%|██████████| 3/3 [00:00<00:00, 35049.89it/s]
Insert of existing embedding ID: 0caadc9e921bc42b049ee99dce67f49a
Insert of existing embedding ID: 8a0697912700ca4e2ac4314c96d7ecf5
Add of existing embedding ID: 0caadc9e921bc42b049ee99dce67f49a
Add of existing embedding ID: 8a0697912700ca4e2ac4314c96d7ecf5
100%|██████████| 12/12 [00:00<00:00, 74126.14it/s]
Insert of existing embedding ID: 0caadc9e921bc42b049ee99dce67f49a
Insert of existing embedding ID: 8a0697912700ca4e2ac4314c96d7ecf5
Insert of existing embe

#### 4. Q&A

In [6]:
rkg_main.answer_question("What program is Mikhail Menshchikov studying for his master's degree?")

Number of requested results 20 is greater than number of elements in index 19, updating n_results = 19
Number of requested results 20 is greater than number of elements in index 19, updating n_results = 19
Number of requested results 20 is greater than number of elements in index 19, updating n_results = 19
Number of requested results 50 is greater than number of elements in index 15, updating n_results = 15


"Mikhail Menshchikov is studying the program 'Deep Learning and Generative AI' for his master's degree at ITMO."

In [7]:
rkg_main.answer_question("Where did Mikhail Menshchikov receive his bachelor's degree?")

Number of requested results 20 is greater than number of elements in index 19, updating n_results = 19
Number of requested results 20 is greater than number of elements in index 19, updating n_results = 19
Number of requested results 50 is greater than number of elements in index 15, updating n_results = 15


'Petrozavodsk State University'